In [46]:
import pandas as pd
import numpy as np
import sklearn as sk
from sklearn import model_selection, linear_model, metrics
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_excel('base_dados_ia.xlsx')
df.head()

,nome,nota1,nota2,frequencia,horas_estudo,resultado
0,Marcos_0,0.3,2.8,64,4,Reprovado
1,Marcos_1,6.8,8.9,55,13,Reprovado
2,Carlos_2,0.3,2.2,82,0,Reprovado
3,Daniel_3,2.0,6.5,84,13,Reprovado
4,Lucas_4,4.5,2.8,50,5,Reprovado


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   nome          120 non-null    str    
 1   nota1         120 non-null    float64
 2   nota2         120 non-null    float64
 3   frequencia    120 non-null    int64  
 4   horas_estudo  120 non-null    int64  
 5   resultado     120 non-null    str    
dtypes: float64(2), int64(2), str(2)
memory usage: 5.8 KB


|Coluna|Tipo de dado|Descrição|
|---|---|---|
|nome|String|Nome do aluno|
|nota1|Float|Primeira nota do aluno|
|nota2|Float|Segunda nota do aluno|
|frequencia|Int|Frequência do aluno em aulas percentual|
|horas_estudo|Int|Horas de estudo|
|resultado|String|Resultado final do aluno (Aprovado/Reprovado)|

---

Com essa rapida análise dos dados, podemos observar que precisamos realizar um tratamento nos mesmos, pois a nossa variável alvo está no formato string e precisamos transformá-la em um formato binario. onde 0 representa reprovado e 1 representa aprovado.

In [19]:
df.sort_values(by='resultado')

,nome,nota1,nota2,frequencia,horas_estudo,resultado
92,Julia_92,8.2,7.5,93,9,Aprovado
30,Felipe_30,8.2,9.8,84,4,Aprovado
94,Felipe_94,6.1,7.7,79,13,Aprovado
99,Paula_99,8.0,6.7,85,10,Aprovado
100,Larissa_100,6.1,7.2,82,13,Aprovado
...,...,...,...,...,...,...
36,Julia_36,7.1,0.6,54,1,Reprovado
35,Ana_35,5.9,2.3,64,0,Reprovado
34,Patrícia_34,4.5,2.5,54,10,Reprovado
46,Julia_46,9.3,8.5,60,13,Reprovado


In [28]:
df_encoded = df.copy()

df_encoded.resultado = df.resultado.map({'Reprovado': 0, 'Aprovado': 1})

In [34]:
X = df_encoded.drop(['nome', 'resultado', 'nota1', 'nota2'], axis=1)
y = df_encoded.resultado

In [35]:
X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(X, y, test_size=0.2, random_state=42)
model = sk.linear_model.LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [39]:
accuracy = sk.metrics.accuracy_score(y_test, y_pred)
print(f'Acurácia: {accuracy:.2f}')

Acurácia: 0.83


In [42]:
df_resultados = X_test.copy()
df_resultados['resultado_real'] = y_test
df_resultados['resultado_predito'] = y_pred
df_resultados

,frequencia,horas_estudo,resultado_real,resultado_predito
44,54,14,0,0
47,53,5,0,0
4,50,5,0,0
55,70,8,0,0
26,98,5,0,0
64,63,8,0,0
73,61,13,0,0
10,79,11,1,0
40,76,14,0,0
107,99,0,0,1


### Com essa primeira visão dos resultados, podemos observar que ele trata a frequencia com um peso muito grande em relação as horas de estudo, vamos normalizar essas variáveis para que o modelo possa aprender melhor a relação entre elas.

In [51]:
df_encoded_normalizado = df_encoded.copy()
df_encoded_normalizado[['frequencia', 'horas_estudo']] = StandardScaler().fit_transform(df_encoded_normalizado[['frequencia', 'horas_estudo']])
df_encoded_normalizado

,nome,nota1,nota2,frequencia,horas_estudo,resultado
0,Marcos_0,0.3,2.8,-0.666980,-0.736709,0
1,Marcos_1,6.8,8.9,-1.261319,1.242509,0
2,Carlos_2,0.3,2.2,0.521697,-1.616361,0
3,Daniel_3,2.0,6.5,0.653773,1.242509,0
4,Lucas_4,4.5,2.8,-1.591508,-0.516796,0
...,...,...,...,...,...,...
115,Camila_115,2.1,3.3,1.578300,1.022596,0
116,Beatriz_116,7.5,8.3,-0.534905,-1.176535,0
117,Felipe_117,0.2,5.4,-0.138679,-0.076970,0
118,Julia_118,7.8,6.5,1.578300,-1.616361,1


### Agora com os dados normalizados vamos treinar essa nova base de dados e observar os resultados.

In [52]:
X = df_encoded_normalizado.drop(['nome', 'resultado', 'nota1', 'nota2'], axis=1)
y = df_encoded_normalizado.resultado
X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(X, y, test_size=0.2, random_state=42)
model = sk.linear_model.LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = sk.metrics.accuracy_score(y_test, y_pred)
print(f'Acurácia: {accuracy:.2f}')

Acurácia: 0.83


### Como podemos observar, a acurácia do modelo foi exatamente a mesma, isso significa que a normalização não influencia modelos de regressão logística, mas pode influenciar outros modelos como redes neurais, SVM, etc.

### A acurácia de 0.83 indica que o modelo está acertando 83% das previsões.